# Gemini LLM Comprehensive Testing

This notebook tests all aspects of Gemini API integration for the Research AI application.

## Test Plan:
1. Environment setup and API key validation
2. Model availability and compatibility testing
3. Basic text generation testing
4. Streaming response testing
5. Error handling and fallback testing
6. Performance and rate limit testing

In [1]:
# Install required packages
!pip install google-generativeai python-dotenv --upgrade


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 5.0 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 5.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.7 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from dotenv import load_dotenv, find_dotenv
import google.generativeai as genai
import time
import json
from typing import List, Dict

# Load environment variables
load_dotenv(find_dotenv())

print("Environment loaded successfully")
print(f"Google Generative AI version: {genai.__version__}")

Environment loaded successfully
Google Generative AI version: 0.8.6


C:\Users\Harold\AppData\Local\Temp\ipykernel_22160\2813411610.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
import google.generativeai as genai

## 1. API Key Validation

In [5]:
def validate_api_key():
    """Validate Gemini API key"""
    api_key = os.getenv('GEMINI_API_KEY')
    
    if not api_key:
        print("❌ No GEMINI_API_KEY found in environment variables")
        return False
    
    print(f"✅ API key found: {api_key[:10]}...")
    
    try:
        genai.configure(api_key=api_key)
        print("✅ API key configured successfully")
        return True
    except Exception as e:
        print(f"❌ Error configuring API key: {e}")
        return False

api_key_valid = validate_api_key()

✅ API key found: AIzaSyBbO_...
✅ API key configured successfully


## 2. Model Availability Testing

In [6]:
def test_model_availability():
    """Test different Gemini model names to find working ones"""
    
    if not api_key_valid:
        print("❌ Cannot test models - invalid API key")
        return []
    
    # List of potential model names to test
    model_names = [
        'gemini-1.5-pro-latest',
        'gemini-1.5-pro',
        'gemini-1.5-flash-latest', 
        'gemini-1.5-flash',
        'gemini-1.0-pro',
        'gemini-pro',
        'gemini-pro-latest',
        'gemini-pro-vision',
        'text-bison-001',
        'chat-bison-001'
    ]
    
    working_models = []
    
    for model_name in model_names:
        try:
            print(f"Testing {model_name}...")
            model = genai.GenerativeModel(model_name)
            response = model.generate_content("Hello")
            print(f"✅ {model_name} works!")
            working_models.append(model_name)
            time.sleep(1)  # Rate limiting
        except Exception as e:
            print(f"❌ {model_name} failed: {str(e)[:100]}...")
    
    return working_models

working_models = test_model_availability()
print(f"\n🎉 Found {len(working_models)} working models: {working_models}")

Testing gemini-1.5-pro-latest...
❌ gemini-1.5-pro-latest failed: 404 models/gemini-1.5-pro-latest is not found for API version v1beta, or is not supported for genera...
Testing gemini-1.5-pro...
❌ gemini-1.5-pro failed: 404 models/gemini-1.5-pro is not found for API version v1beta, or is not supported for generateConte...
Testing gemini-1.5-flash-latest...
❌ gemini-1.5-flash-latest failed: 404 models/gemini-1.5-flash-latest is not found for API version v1beta, or is not supported for gene...
Testing gemini-1.5-flash...
❌ gemini-1.5-flash failed: 404 models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateCon...
Testing gemini-1.0-pro...
❌ gemini-1.0-pro failed: 404 models/gemini-1.0-pro is not found for API version v1beta, or is not supported for generateConte...
Testing gemini-pro...
❌ gemini-pro failed: 404 models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. ...
Testing gemini-pro-latest...
❌ gemini-pro-

## 3. List Available Models (API Method)

In [7]:
def list_available_models():
    """List all available models using the API"""
    
    if not api_key_valid:
        print("❌ Cannot list models - invalid API key")
        return
    
    try:
        models = genai.list_models()
        print("📋 Available models:")
        
        for model in models:
            print(f"  - {model.name}")
            print(f"    Display name: {model.display_name}")
            print(f"    Description: {model.description}")
            print(f"    Generation methods: {model.supported_generation_methods}")
            print()
            
        return models
    except Exception as e:
        print(f"❌ Error listing models: {e}")
        return []

available_models = list_available_models()

📋 Available models:
  - models/gemini-2.5-flash
    Display name: Gemini 2.5 Flash
    Description: Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.
    Generation methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']

  - models/gemini-2.5-pro
    Display name: Gemini 2.5 Pro
    Description: Stable release (June 17th, 2025) of Gemini 2.5 Pro
    Generation methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']

  - models/gemini-2.0-flash
    Display name: Gemini 2.0 Flash
    Description: Gemini 2.0 Flash
    Generation methods: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']

  - models/gemini-2.0-flash-001
    Display name: Gemini 2.0 Flash 001
    Description: Stable version of Gemini 2.0 Flash, our fast and versatile multimodal model for scaling across diverse tasks, released in January of 2025

## 4. Basic Text Generation Testing

In [ ]:
def test_basic_generation(model_name: str):
    """Test basic text generation with a model"""
    
    if not api_key_valid:
        print("❌ Cannot test generation - invalid API key")
        return
    
    try:
        print(f"🧪 Testing basic generation with {model_name}")
        model = genai.GenerativeModel(model_name)
        
        # Test simple prompt
        prompt = "Write a brief summary of artificial intelligence in 2-3 sentences."
        response = model.generate_content(prompt)
        
        print(f"✅ Generation successful!")
        print(f"Response: {response.text}")
        print(f"Tokens used: {response.usage_metadata.total_token_count if hasattr(response, 'usage_metadata') else 'N/A'}")
        
        return True
    except Exception as e:
        print(f"❌ Generation failed: {e}")
        return False

# Test with the first working model
if working_models:
    test_basic_generation(working_models[0])
else:
    print("❌ No working models available for testing")

## 5. Streaming Response Testing

In [ ]:
def test_streaming_generation(model_name: str):
    """Test streaming text generation"""
    
    if not api_key_valid:
        print("❌ Cannot test streaming - invalid API key")
        return
    
    try:
        print(f"🌊 Testing streaming generation with {model_name}")
        model = genai.GenerativeModel(model_name)
        
        prompt = "Explain the concept of machine learning step by step."
        response = model.generate_content(prompt, stream=True)
        
        print("Streaming response:")
        full_text = ""
        for chunk in response:
            if chunk.text:
                print(chunk.text, end='', flush=True)
                full_text += chunk.text
        
        print(f"\n\n✅ Streaming completed!")
        print(f"Total length: {len(full_text)} characters")
        
        return True
    except Exception as e:
        print(f"❌ Streaming failed: {e}")
        return False

# Test streaming with the first working model
if working_models:
    test_streaming_generation(working_models[0])
else:
    print("❌ No working models available for streaming test")

## 6. Research-Specific Prompt Testing

In [ ]:
def test_research_prompts(model_name: str):
    """Test prompts similar to what Research AI would use"""
    
    if not api_key_valid or not working_models:
        print("❌ Cannot test research prompts - no valid setup")
        return
    
    model = genai.GenerativeModel(model_name)
    
    # Test preview summary prompt
    preview_prompt = (
        "Write a concise single-paragraph summary about 'climate change' (5-6 sentences).\n\n"
        "Content:\nClimate change refers to long-term shifts in global temperatures and weather patterns. "
        "While climate variations are natural, human activities have been the main driver since the mid-20th century. "
        "The burning of fossil fuels, deforestation, and industrial processes release greenhouse gases."
    )
    
    print("🔬 Testing preview summary prompt...")
    try:
        response = model.generate_content(preview_prompt)
        print(f"✅ Preview summary generated:")
        print(response.text)
        print()
    except Exception as e:
        print(f"❌ Preview summary failed: {e}")
    
    # Test detailed summary prompt
    detailed_prompt = (
        "Summarize 'artificial intelligence' in 7 to 9 paragraphs with intro, body, and conclusion.\n\n"
        "Content:\nArtificial intelligence (AI) is intelligence demonstrated by machines, "
        "in contrast to the natural intelligence displayed by humans and animals. "
        "AI research has been defined as the field of study of intelligent agents, "
        "which refers to any device that perceives its environment and takes actions "
        "that maximize its chance of successfully achieving its goals."
    )
    
    print("📚 Testing detailed summary prompt...")
    try:
        response = model.generate_content(detailed_prompt)
        print(f"✅ Detailed summary generated (first 200 chars):")
        print(response.text[:200] + "..." if len(response.text) > 200 else response.text)
        print()
    except Exception as e:
        print(f"❌ Detailed summary failed: {e}")
    
    # Test training-only prompt
    training_prompt = (
        "Write a concise, single-paragraph summary about 'quantum computing' based on your knowledge. "
        "Keep it brief and to the point – just one paragraph, no more than 5-6 sentences."
    )
    
    print("🧠 Testing training-only prompt...")
    try:
        response = model.generate_content(training_prompt)
        print(f"✅ Training-only summary generated:")
        print(response.text)
        print()
    except Exception as e:
        print(f"❌ Training-only summary failed: {e}")

if working_models:
    test_research_prompts(working_models[0])
else:
    print("❌ No working models available for research prompt testing")

## 7. Error Handling and Edge Cases

In [ ]:
def test_error_handling():
    """Test various error scenarios"""
    
    if not api_key_valid:
        print("❌ Cannot test error handling - invalid API key")
        return
    
    print("🧪 Testing error handling scenarios...")
    
    # Test invalid model name
    try:
        print("Testing invalid model name...")
        model = genai.GenerativeModel('invalid-model-name')
        response = model.generate_content("test")
        print("❌ Expected error but got success")
    except Exception as e:
        print(f"✅ Correctly caught invalid model error: {str(e)[:100]}...")
    
    # Test empty prompt
    try:
        print("Testing empty prompt...")
        if working_models:
            model = genai.GenerativeModel(working_models[0])
            response = model.generate_content("")
            print(f"✅ Empty prompt handled: {response.text[:50]}...")
    except Exception as e:
        print(f"✅ Empty prompt error handled: {str(e)[:100]}...")
    
    # Test very long prompt
    try:
        print("Testing very long prompt...")
        if working_models:
            model = genai.GenerativeModel(working_models[0])
            long_prompt = "test " * 10000  # Very long prompt
            response = model.generate_content(long_prompt)
            print(f"✅ Long prompt handled: {len(response.text)} chars")
    except Exception as e:
        print(f"✅ Long prompt error handled: {str(e)[:100]}...")

test_error_handling()

## 8. Performance Testing

In [ ]:
def test_performance():
    """Test response times and performance"""
    
    if not api_key_valid or not working_models:
        print("❌ Cannot test performance - no valid setup")
        return
    
    model = genai.GenerativeModel(working_models[0])
    
    # Test response times
    prompts = [
        "What is 2+2?",
        "Explain photosynthesis in one sentence.",
        "Write a haiku about programming.",
        "Summarize the history of computers in 3 sentences."
    ]
    
    print("⏱️ Testing performance...")
    
    for i, prompt in enumerate(prompts, 1):
        try:
            start_time = time.time()
            response = model.generate_content(prompt)
            end_time = time.time()
            
            response_time = end_time - start_time
            response_length = len(response.text)
            
            print(f"Prompt {i}: {response_time:.2f}s, {response_length} chars")
            time.sleep(1)  # Rate limiting
        except Exception as e:
            print(f"Prompt {i} failed: {e}")

test_performance()

## 9. Summary and Recommendations

In [ ]:
def generate_summary_report():
    """Generate a summary report of all tests"""
    
    print("📊 GEMINI LLM TEST SUMMARY")
    print("=" * 50)
    print(f"API Key Status: {'✅ Valid' if api_key_valid else '❌ Invalid'}")
    print(f"Working Models: {len(working_models)} found")
    
    if working_models:
        print(f"Best Model: {working_models[0]}")
        print("All Working Models:")
        for model in working_models:
            print(f"  - {model}")
    
    print("\n🎯 Recommendations:")
    
    if api_key_valid and working_models:
        print("✅ Gemini API is ready for integration")
        print(f"✅ Use '{working_models[0]}' as the primary model")
        print("✅ Implement fallback to other working models")
        print("✅ Add proper error handling for API limits")
    elif api_key_valid:
        print("⚠️ API key valid but no working models found")
        print("⚠️ Check API permissions and model access")
    else:
        print("❌ API key not configured")
        print("❌ Set up GEMINI_API_KEY in environment variables")
    
    print("\n🔧 Integration Steps:")
    print("1. Add model discovery and fallback logic")
    print("2. Implement proper error handling")
    print("3. Add rate limiting and retry logic")
    print("4. Test with Research AI prompts")
    print("5. Integrate with existing GUI")

generate_summary_report()